<a href="https://colab.research.google.com/github/ynam0327-afk/REDRED/blob/main/model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import joblib
from google.colab import drive
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [5]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV
)

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

from xgboost import XGBClassifier


# ==================================
# 1. 데이터 불러오기
# ==================================

df = pd.read_csv("/content/drive/MyDrive/REDRED/url_features.csv")

feature_cols = [
    "url_length",
    "domain_length",
    "dot_count",
    "slash_count",
    "hyphen_count",
    "digit_count",
    "contains_at",
    "has_ip",
    "https_flag",
    "shortener",
    "suspicious_word_count"
]

X = df[feature_cols]
y = df["label"]


# ==================================
# 2. Train/Test 분리
# ==================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train :", X_train.shape)
print("Test  :", X_test.shape)


# ==================================
# 3. Random Forest 학습
# ==================================

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n========================")
print("Random Forest")
print("========================")

print("Accuracy :", round(accuracy_score(y_test, rf_pred), 4))
print("Precision:", round(precision_score(y_test, rf_pred), 4))
print("Recall   :", round(recall_score(y_test, rf_pred), 4))
print("F1-score :", round(f1_score(y_test, rf_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, rf_prob), 4))

print("\nClassification Report")
print(classification_report(y_test, rf_pred))


# ==================================
# 4. XGBoost 하이퍼파라미터 튜닝
# ==================================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1]
}

xgb = XGBClassifier(
    eval_metric="logloss",
    random_state=42
)

grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\n최적 파라미터")
print(grid_search.best_params_)

best_xgb = grid_search.best_estimator_


# ==================================
# 5. XGBoost 평가
# ==================================

xgb_pred = best_xgb.predict(X_test)
xgb_prob = best_xgb.predict_proba(X_test)[:, 1]

print("\n========================")
print("XGBoost")
print("========================")

print("Accuracy :", round(accuracy_score(y_test, xgb_pred), 4))
print("Precision:", round(precision_score(y_test, xgb_pred), 4))
print("Recall   :", round(recall_score(y_test, xgb_pred), 4))
print("F1-score :", round(f1_score(y_test, xgb_pred), 4))
print("ROC-AUC  :", round(roc_auc_score(y_test, xgb_prob), 4))

print("\nClassification Report")
print(classification_report(y_test, xgb_pred))


# ==================================
# 6. 중요 변수 확인
# ==================================

rf_importance = pd.Series(
    rf_model.feature_importances_,
    index=feature_cols
).sort_values(ascending=False)

print("\nRandom Forest Feature Importance")
print(rf_importance)


# ==================================
# 7. 모델 저장
# ==================================

joblib.dump(rf_model, "/content/drive/MyDrive/REDRED/rf_url_model.joblib")
joblib.dump(best_xgb, "/content/drive/MyDrive/REDRED/xgb_url_model.joblib")

print("\n모델 저장 완료")

Train : (520952, 11)
Test  : (130239, 11)

Random Forest
Accuracy : 0.9263
Precision: 0.8833
Recall   : 0.9044
F1-score : 0.8937
ROC-AUC  : 0.9749

Classification Report
              precision    recall  f1-score   support

           0       0.95      0.94      0.94     85621
           1       0.88      0.90      0.89     44618

    accuracy                           0.93    130239
   macro avg       0.92      0.92      0.92    130239
weighted avg       0.93      0.93      0.93    130239

Fitting 3 folds for each of 8 candidates, totalling 24 fits

최적 파라미터
{'learning_rate': 0.1, 'max_depth': 6, 'n_estimators': 200}

XGBoost
Accuracy : 0.9008
Precision: 0.8585
Recall   : 0.8506
F1-score : 0.8545
ROC-AUC  : 0.9623

Classification Report
              precision    recall  f1-score   support

           0       0.92      0.93      0.92     85621
           1       0.86      0.85      0.85     44618

    accuracy                           0.90    130239
   macro avg       0.89      0.89 